In [1]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import zscore
from scipy.stats import norm
from scipy import stats
import scanpy as sc
import rapids_singlecell as rsc
import anndata
import os
from scipy.stats import gaussian_kde

/home/labuser/anaconda3/envs/rapids-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import pandas as pd

# === USER-DEFINED PATHS ===
# This should be the directory that contains the per-slide subfolders (e.g., B004-A-004, B004-A-008, ...)
input_root = "/mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/"
output_dir  = "/mnt/jwh83-data/Confetti/output/Redsea/Clustering"
os.makedirs(output_dir, exist_ok=True)

before_out = os.path.join(output_dir, "CellSAM_single_cell_before_redsea_ALL.csv")
after_out  = os.path.join(output_dir, "CellSAM_single_cell_after_redsea_ALL.csv")

# Filenames inside each subfolder
BEFORE_NAME = "single_cell_before_redsea.csv"
AFTER_NAME  = "single_cell_after_redsea.csv"

before_list, after_list = [], []

def load_and_tag(csv_path: str, slide_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Add slide_name column (parent folder name)
    df["slide_name"] = slide_name

    # Update CellID -> "{slide_name}_{original}"
    if "CellID" in df.columns:
        df["CellID"] = slide_name + "_" + df["CellID"].astype(str)
    else:
        print(f"⚠️ Missing 'CellID' column in: {csv_path}")

    # Optional: keep provenance
    df["FILE"] = os.path.basename(csv_path)

    return df

# === ITERATE SUBFOLDERS ===
for entry in sorted(os.listdir(input_root)):
    slide_dir = os.path.join(input_root, entry)
    if not os.path.isdir(slide_dir):
        continue

    slide_name = entry  # parent subfolder name

    before_path = os.path.join(slide_dir, BEFORE_NAME)
    after_path  = os.path.join(slide_dir, AFTER_NAME)

    if os.path.exists(before_path):
        try:
            print(f"Processing BEFORE: {before_path}")
            before_list.append(load_and_tag(before_path, slide_name))
        except Exception as e:
            print(f"⚠️ Error processing {before_path}: {e}")
    else:
        print(f"⚠️ Missing BEFORE file for {slide_name}: {before_path}")

    if os.path.exists(after_path):
        try:
            print(f"Processing AFTER:  {after_path}")
            after_list.append(load_and_tag(after_path, slide_name))
        except Exception as e:
            print(f"⚠️ Error processing {after_path}: {e}")
    else:
        print(f"⚠️ Missing AFTER file for {slide_name}: {after_path}")

# === COMBINE & SAVE ===
if before_list:
    before_df = pd.concat(before_list, ignore_index=True)
    before_df.to_csv(before_out, index=False)
    print(f"✅ Combined BEFORE CSV saved to: {before_out}")
else:
    print("❌ No BEFORE CSV files were processed.")

if after_list:
    after_df = pd.concat(after_list, ignore_index=True)
    after_df.to_csv(after_out, index=False)
    print(f"✅ Combined AFTER CSV saved to: {after_out}")
else:
    print("❌ No AFTER CSV files were processed.")


Processing BEFORE: /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-004/single_cell_before_redsea.csv
Processing AFTER:  /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-004/single_cell_after_redsea.csv
Processing BEFORE: /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-008/single_cell_before_redsea.csv
Processing AFTER:  /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-008/single_cell_after_redsea.csv
Processing BEFORE: /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-104/single_cell_before_redsea.csv
Processing AFTER:  /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-104/single_cell_after_redsea.csv
Processing BEFORE: /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-204/single_cell_before_redsea.csv
Processing AFTER:  /mnt/jwh83-data/Confetti/output/Redsea/20260127_ns444_CellSAM/B004-A-204/single_cell_after_redsea.csv
Processing BEFORE: /mnt/jwh8